In [ ]:
# !pip install uv

In [1]:
!uv pip install openai==2.31.0 pandas==2.3.3 openpyxl==3.1.5 arize==8.11.0 arize-otel==0.12.0 openinference-instrumentation-openai==0.1.44 openinference-instrumentation-openai-agents==1.4.1 openai-agents==0.13.6 python-dotenv==1.2.2

Audited 7 packages in 39ms


In [ ]:
# !git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting)
Add the following variables in the Secrets section:
- `ARIZE_API_KEY`
- `ARIZE_SPACE_ID`
- `OPENAI_API_KEY`

In [2]:
from pathlib import Path
import json
import sys

import pandas as pd
pd.set_option('max_colwidth', None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'workshop_helpers').exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / 'workshop_helpers').exists()]
    if not candidates:
        raise FileNotFoundError('Could not find a repo folder containing workshop_helpers')
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import TOOLS, hydrate_backend_from_dataset, snapshot_backend
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    VARIANT_BEHAVIORS,
    build_review_context_message,
    dispatch_specialist_response,
    ensure_arize_dataset,
    format_checklist_rows,
    prepare_experiment_bundle,
    production_readiness_checklist,
    run_experiment,
    run_router_local,
    select_cases,
    select_cases_by_category,
    summarize_dataset,
    summarize_experiment_scores,
    summarize_router_experiment_results,
)
from workshop_helpers.metrics import (
    RoutingAccuracyEvaluator,
    escalation_decision_result,
    judge_brand_voice,
    judge_helpfulness,
    pack_response_payload,
    score_routing_response,
)
from workshop_helpers.scenarios import run_router_structured

In [3]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env['client']
ARIZE_SPACE_ID = env['arize_space_id']
ARIZE_API_KEY = env['arize_api_key']

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: Workiva-ProblemFirst-Workshop-2026-04-11
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



# Workiva - AI Support Copilot Workshop

We will work through two systems:
1. `v1` router classification with a code-based exact-match metric.
2. `v2` routed draft-reply creation with three specialist behaviors:
   - `permissions`: single LLM call
   - `review_workflow`: LLM call with injected workflow context
   - `billing`: tool-using billing agent

The routing approach is deterministic structured output routing.
The router selects a support category, then the notebook dispatches to the matching specialist.

## Setup

The scenario library has 50 cases total.
Use `LIMIT_N_CASES` for a faster workshop run if needed.

In [4]:
LIMIT_N_CASES = 10  # Set to an integer like 10 for a faster run. For full dataset, set to None.
EXPERIMENT_DATASET = select_cases(DATASET, limit_n=LIMIT_N_CASES)
library_summary = summarize_dataset(DATASET)
experiment_summary = summarize_dataset(EXPERIMENT_DATASET)
backend_snapshot = hydrate_backend_from_dataset(DATASET)

print(f"Scenario library: {library_summary['scenario_count']} total cases")
print(f"Experiment dataset: {experiment_summary['scenario_count']} cases")
print(f"Categories: {', '.join(experiment_summary['categories'])}")
print(f"Limit applied: {LIMIT_N_CASES}")
print()
print('Demo backend state:')
for label, value in backend_snapshot.items():
    print(f'  {label}: {value}')

pd.DataFrame(EXPERIMENT_DATASET)[['scenario_id', 'category', 'difficulty', 'is_edge_case']].head(12)

Scenario library: 50 total cases
Experiment dataset: 10 cases
Categories: billing, escalation, permissions, review_workflow
Limit applied: 10

Demo backend state:
  billing_account_count: 22
  invoice_count: 14
  billing_reference_topics: 6


,scenario_id,category,difficulty,is_edge_case
0,CR_012,review_workflow,standard,False
1,CR_010,review_workflow,standard,False
2,CR_006,review_workflow,standard,False
3,CP_012,permissions,standard,False
4,CP_005,permissions,standard,False
5,CE_004,escalation,edge,True
6,CR_013,review_workflow,standard,False
7,CP_010,permissions,standard,False
8,CB_002,billing,standard,False
9,CR_003,review_workflow,standard,False


### Create The Arize Dataset

Create the dataset once during setup so you can inspect it in Arize before running experiments.

In [ ]:
ARIZE_DATASET_BUNDLE = ensure_arize_dataset(
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=EXPERIMENT_DATASET,
)
arize_client = ARIZE_DATASET_BUNDLE['client']
DATASET_ID = ARIZE_DATASET_BUNDLE['dataset_id']
ROUTER_DATASET_ID = DATASET_ID

print(f"Arize dataset: {ARIZE_DATASET_BUNDLE['dataset_name']}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Rows uploaded: {ARIZE_DATASET_BUNDLE['row_count']}")
print(f"Created in this run: {ARIZE_DATASET_BUNDLE['created']}")

display(ARIZE_DATASET_BUNDLE['dataframe'].head(10))


In [5]:
PERMISSIONS_CASE = select_cases_by_category(DATASET, 'permissions')[0]
REVIEW_CASE = select_cases_by_category(DATASET, 'review_workflow')[0]
BILLING_CASE = select_cases_by_category(DATASET, 'billing')[0]
ESCALATION_CASE = select_cases_by_category(DATASET, 'escalation')[0]
ROUTING_CATEGORIES = sorted({case['category'] for case in DATASET})
print(f"The available routing categories: {ROUTING_CATEGORIES}")

# We are picking a billing case for the demo
ROUTER_DEMO_CASE = BILLING_CASE

The available routing categories: ['billing', 'escalation', 'permissions', 'review_workflow']


---
## Part 1 - `v1` Routing Classification

We start with the simplest system: a router that maps each user message to one support category.

The loop is:
1. Run the router once.
2. Add a code-based exact-match eval.
3. Improve the prompt.
4. Re-run and compare the score.

In [6]:
PROMPT_ROUTER_BASELINE = (
    'You are a support routing classifier. '
)

router_baseline = run_router_structured(
    client,
    ROUTER_DEMO_CASE['user_input'],
    PROMPT_ROUTER_BASELINE.format(categories=', '.join(ROUTING_CATEGORIES)),
    ROUTING_CATEGORIES,
)
router_payload_baseline = pack_response_payload(router_baseline['category'], metadata=router_baseline)

display(pd.DataFrame([
    {
        'scenario_id': ROUTER_DEMO_CASE['scenario_id'],
        'user_input': ROUTER_DEMO_CASE['user_input'],
        'expected_category': ROUTER_DEMO_CASE['category'],
        'predicted_category': router_baseline['category'],
        **score_routing_response(router_payload_baseline, ROUTER_DEMO_CASE['category']),
    }
]))
print(json.dumps(router_baseline, indent=2))

,scenario_id,user_input,expected_category,predicted_category,exact_match,exact_match_reasoning,total
0,CB_001,I was billed $49 twice this month. Can you fix that?,billing,escalation,Poor,Predicted `escalation` but expected `billing`.,0.0


{
  "category": "escalation",
  "fallback_reason": "Invalid category returned: I can\u2019t assist with billing issues directly. Please contact customer support for help with your billing inquiry."
}


### Run The Router Experiment In Arize

Now that the dataset exists in Arize, we can run the router experiment and keep the traces and scores tied to that same dataset.
We start by running the baseline prompt across the dataset.


In [7]:
def task_router_baseline(row: dict) -> str:
    route = run_router_structured(
        client,
        row['user_input'],
        PROMPT_ROUTER_BASELINE.format(categories=', '.join(ROUTING_CATEGORIES)),
        ROUTING_CATEGORIES,
    )
    return route['category']

print('Running router baseline experiment without evaluators...')
run_router_baseline_traces = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-baseline-traces-only',
    task=task_router_baseline,
    evaluators=[],
)

print(f"Trace-only router experiment ID: {run_router_baseline_traces['experiment'].id}")
print(f"Rows evaluated: {len(run_router_baseline_traces['results_df'])}")

  arize.pre_releases | WARNING | [BETA] datasets.list is a beta API in Arize SDK v8.11.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.
Running router baseline experiment without evaluators...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


I0411 16:47:16.282852 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283329 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283346 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283348 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283349 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283351 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283353 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283354 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283355 19905773 ev_poll_posix.cc:593] FD from fork parent still i

running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

n poll list: fd(98, generation: 1)
I0411 16:47:16.283614 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283615 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283616 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283618 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283619 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283620 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283621 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283622 19905773 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0411 16:47:16.283623 19905773 ev_poll_posix.

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/11/26 04:47 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
  arize.pre_releases | WARNING | [BETA] experiments.get is a beta API in Arize SDK v8.11.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.
Trace-only router experiment ID: RXhwZXJpbWVudDo3NjEyNTpuNGND
Rows evaluated: 10


### Add The Evaluator And Re-Run

* We now have traces. But, how well is our system performing?
* To answer that question, we add an Evaluator. In this case, we are going to use the exact-match evaluator.
* This turns the full-dataset run from trace inspection into a measurable benchmark.


In [8]:
router_eval = [RoutingAccuracyEvaluator(expected_field='category')]

print('Running router baseline experiment with exact-match evaluator...')
run_router_baseline = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-baseline',
    task=task_router_baseline,
    evaluators=router_eval,
)

router_baseline_summary = summarize_router_experiment_results(run_router_baseline['results_df'])
display(router_baseline_summary)

Running router baseline experiment with exact-match evaluator...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/11/26 04:48 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


,rows_evaluated,score_column,exact_match_accuracy
0,10,eval.RoutingAccuracyEvaluator.score,0.1


### Fix The Prompt And Re-Run The Same Experiment

Once we have the baseline score, we improve the router prompt and run the same Arize experiment flow again.
That lets us verify whether the score actually improved.

Couple of tools available for prompt optimization:
* From OpenAI: https://developers.openai.com/api/docs/guides/prompt-optimizer
* From Arize: https://arize.com/docs/ax/prompts/prompt-optimization
* From Arize: Using Alyx

### Let's try Alyx here.

Go to the experiment that we just ran. Click on **Ask Alyx** button. Enter the following:

_The system prompt used for this experiment was "You are a support routing classifier.". Looking at the RoutingAccuracyEvaluator, are you able to help suggest an improvement in the system prompt. Also, I would like to receive a json response with the value in the key 'category'._


In [9]:
# Hand written
# PROMPT_ROUTER = (
#     'You are a precise support routing classifier for an AI support copilot. '
#     'Classify each user question into exactly one category from this list: permissions, review_workflow, billing, escalation. '
#     'Choose permissions for access, sharing, ownership, or admin-rights requests. '
#     'Choose review_workflow for approval status, blockers, sign-off, resubmission, or workflow-state questions. '
#     'Choose billing for invoices, charges, credits, taxes, pricing, payment methods, renewals, or refunds. '
#     'Choose escalation for urgent, high-risk, angry, security-sensitive, legal-risk, or explicitly human-handoff requests. '
#     'Return strict JSON with one key: category.'
# )

# From Alyx
PROMPT_ROUTER = '''You are a support routing classifier. Your task is to classify customer support queries into one of the following categories and return ONLY a JSON response.

**Valid Categories:**
- review_workflow: Questions about file status, approval workflows, review steps, pending actions, or workflow exceptions
- permissions: Questions about access rights, sharing permissions, role-based access, or authorization
- billing: Questions about invoices, charges, pricing, or payment issues
- escalation: Urgent issues involving security breaches, sensitive data exposure, or critical problems requiring immediate human attention

**Instructions:**
1. Analyze the user's input to determine which category best fits their query
2. Return ONLY a JSON object with the key 'category' and the appropriate category value
3. Do NOT provide explanations or additional text in your response

**Output Format:**
{"category": "review_workflow"}

**Examples:**
- "How do I know if the file is waiting on me?" → {"category": "review_workflow"}
- "Can contractors access this document?" → {"category": "permissions"}
- "Why is my invoice higher this month?" → {"category": "billing"}
- "We shared sensitive data with the wrong party" → {"category": "escalation"}
'''

def task_router_improved(row: dict) -> str:
    route = run_router_structured(
        client,
        row['user_input'],
        PROMPT_ROUTER,
        ROUTING_CATEGORIES,
    )
    return route['category']

print('Running router improved experiment with exact-match evaluator...')
run_router_improved = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-improved',
    task=task_router_improved,
    evaluators=router_eval,
)

router_improved_summary = summarize_router_experiment_results(run_router_improved['results_df'])

router_comparison = pd.DataFrame([
    {
        'variant': 'router_baseline',
        'exact_match_accuracy': router_baseline_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_baseline_summary.loc[0, 'rows_evaluated'],
    },
    {
        'variant': 'router_improved_prompt',
        'exact_match_accuracy': router_improved_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_improved_summary.loc[0, 'rows_evaluated'],
    },
])

display(router_comparison)

Running router improved experiment with exact-match evaluator...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/11/26 04:49 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


,variant,exact_match_accuracy,rows_evaluated
0,router_baseline,0.1,10
1,router_improved_prompt,1.0,10


---
## Part 2 - `v2` Routed Draft Replies

`v2` keeps the same router, but now the routed category determines how we answer.

We are not building the specialists from scratch.
The session focuses is on understanding the behavior, then evaluating reliability.

The specialist implementations are intentionally different:
- `permissions`: prompt-only
- `review_workflow`: prompt plus injected context
- `billing`: tool-using ReAct-style agent
- `escalation`: human handoff response

In [10]:
PROMPT_PERMISSIONS = (
    'You are a permissions support specialist. '
    'Answer access questions clearly and concisely. '
    'Explain the right request path, who can grant access, and any policy limitation. '
    'Do not claim you changed permissions or approved access.'
)

PROMPT_REVIEW_WORKFLOW = (
    'You are a review-workflow support specialist. '
    'Use the provided workflow context to explain the current stage, blockers, and next best step. '
    'Do not invent missing approvers or workflow states.'
)

PROMPT_BILLING = (
    'You are a billing support specialist for an AI support copilot. '
    'The authenticated billing account ID for this session is: {authenticated_account_id}. '
    'Use get_billing_account and get_invoice_details to verify facts before replying. '
    'Use read_billing_reference for policy guidance. '
    'If a billing credit or human handoff is required, take the action before replying. '
    'Use apply_billing_credit or escalate_to_human when appropriate.'
)

ESCALATION_RESPONSE_TEMPLATE = (
    "I'm sorry this requires urgent human attention. "
    "I've escalated the case for {account_name} to a human specialist, and they will follow up directly."
)


In [11]:
def run_v2_routed_demo(case: dict):
    route = run_router_structured(
        client,
        case['user_input'],
        PROMPT_ROUTER,
        ROUTING_CATEGORIES,
    )
    payload = dispatch_specialist_response(
        client=client,
        route_category=route['category'],
        case=case,
        prompt_permissions=PROMPT_PERMISSIONS,
        prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
        prompt_billing=PROMPT_BILLING,
        escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    )
    return route, json.loads(payload)

specialist_patterns = pd.DataFrame([
    {
        'category': 'permissions',
        'pattern': 'single LLM call',
        'model_input_preview': PERMISSIONS_CASE['user_input'],
    },
    {
        'category': 'review_workflow',
        'pattern': 'LLM call with injected workflow context',
        'model_input_preview': build_review_context_message(REVIEW_CASE['user_input'], REVIEW_CASE['source_data']),
    },
    {
        'category': 'billing',
        'pattern': 'tool-using billing agent',
        'model_input_preview': BILLING_CASE['user_input'],
    },
])

print('Billing tools available to the routed system:')
for tool in TOOLS:
    print(f'  - {tool.name}')

print('\nWhat each specialist sees:')
display(specialist_patterns)


Billing tools available to the routed system:
  - get_billing_account
  - get_invoice_details
  - read_billing_reference
  - apply_billing_credit
  - escalate_to_human

What each specialist sees:


,category,pattern,model_input_preview
0,permissions,single LLM call,How can I get permission to access the Q2 Board deck? I only have view access right now.
1,review_workflow,LLM call with injected workflow context,"Workflow context:\n{\n ""file_name"": ""Q2 disclosure memo"",\n ""current_stage"": ""legal review"",\n ""completed_approvals"": [\n ""Finance""\n ],\n ""pending_approvals"": [\n ""Legal""\n ],\n ""blocking_reason"": ""Legal approval still pending"",\n ""next_step"": ""Wait for Legal or reassign the approver if they are unavailable.""\n}\n\nUser question:\nWhat is blocking this file from moving forward? I already got one approval."
2,billing,tool-using billing agent,I was billed $49 twice this month. Can you fix that?


In [13]:
def summarize_v2_case(case: dict) -> dict:
    route, payload = run_v2_routed_demo(case)
    return {
        'scenario_id': case['scenario_id'],
        'expected_category': case['category'],
        'predicted_category': route['category'],
        'response_text': payload['response_text'],
    }

specialist_demo = pd.DataFrame([
    summarize_v2_case(PERMISSIONS_CASE),
    summarize_v2_case(REVIEW_CASE),
    summarize_v2_case(BILLING_CASE),
    summarize_v2_case(ESCALATION_CASE),
])

display(specialist_demo[['scenario_id', 'expected_category', 'predicted_category', 'response_text']])

,scenario_id,expected_category,predicted_category,response_text
0,CP_001,permissions,permissions,"To request access to the Q2 Board deck, you should follow these steps:\n\n1. **Identify the Right Request Path**: Submit a formal access request through your organization's access request system or contact your direct supervisor.\n\n2. **Who Can Grant Access**: Typically, access to sensitive documents like the Q2 Board deck can be granted by your department head or the data owner responsible for the Board materials.\n\n3. **Policy Limitations**: Be aware that access may be restricted based on your role and the sensitivity of the information. Ensure you have a valid reason for needing enhanced access and be prepared to explain how it aligns with your responsibilities.\n\nMake sure to follow your organization's policies and procedures when submitting your request."
1,CR_001,review_workflow,review_workflow,"The current stage of the workflow for the ""Q2 disclosure memo"" is legal review. While you have successfully obtained approval from the Finance team, the file is currently blocked because it is still pending approval from the Legal team. Until Legal completes their review and provides their approval, the file cannot move forward.\n\nThe next best step is to wait for the Legal team to complete their review. If you find that they are unavailable or taking longer than expected, you may consider reassigning the approver to expedite the process."
2,CB_001,billing,billing,"I've applied a billing credit of $49 to your account for the duplicate invoice. This credit should be visible on your account within 1 business day.\n\nAdditionally, your case has been escalated to a human billing specialist for further investigation, and you can expect a follow-up within the next 2 business hours.\n\nIf you have any other questions or need further assistance, feel free to ask!"
3,CE_001,escalation,escalation,"I'm sorry this requires urgent human attention. I've escalated the case for Redwood Equity to a human specialist, and they will follow up directly."


### Run The Full `v2` System In Arize

As with the full-dataset `v1` run, the main benchmark happens in Arize.
We start by running the routed copilot without evaluators so we can inspect traces first.


In [14]:
experiment_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts={},
    limit_n=LIMIT_N_CASES,
    arize_client=arize_client,
    dataset_id=DATASET_ID,
)
arize_client = experiment_bundle['client']
DATASET_ID = experiment_bundle['dataset_id']
tasks = experiment_bundle['tasks']

print('Running routed copilot experiment without evaluators...')
run_v2_routed_traces = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix='v2-routed-copilot-traces-only',
    task=tasks['task_v2_routed'],
    evaluators=[],
    concurrency=1,
)

print(f"Trace-only routed experiment ID: {run_v2_routed_traces['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_routed_traces['results_df'])}")

Running routed copilot experiment without evaluators...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/11/26 04:53 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
Trace-only routed experiment ID: RXhwZXJpbWVudDo3NjEyODpLSWtz
Rows evaluated: 10


### Trace Annotation And Error Analysis

At this point we have routed traces, but not quality scores yet.
This is the right moment to review examples, annotate patterns, and decide what we want the LLM judges to measure.


### Error Analysis Prompts

Useful questions:
- Which routed replies are strong on helpfulness but weak on brand voice?
- Which escalation misses are routing errors versus specialist behavior errors?
- Which billing traces used the wrong tools or skipped the required action?
- Which patterns should we encode into the judge prompts? 


### Add The `v2` Evaluators And Re-Run

Now we define the `v2` evaluators, then rerun the same Arize experiment flow with scoring turned on.

The evaluator set is:
- `brand_voice`: LLM judge
- `helpfulness`: LLM judge
- `escalation decision`: code eval for whether the system correctly triggers `escalate_to_human` when required
- `expected action taken`: code eval for whether the expected action, such as `apply_billing_credit` or `escalate_to_human`, was actually recorded in the action calls

In the helper code, the two code evaluators are implemented as `EscalationDecisionEvaluator` and `ExpectedActionTakenEvaluator`, so those names may show up in Arize.


In [15]:
JUDGE_PROMPTS = {
    'brand_voice': (
        'You evaluate AI support responses for brand voice.\n\n'
        'Return exactly two lines in this format:\n'
        'LABEL: <Good|Acceptable|Poor>\n'
        'REASONING: <one sentence>\n\n'
        'GOOD: Calm, professional, concise, and empathetic without sounding robotic.\n'
        'ACCEPTABLE: Mostly professional but generic, awkward, or slightly flat.\n'
        'POOR: Blaming, abrupt, defensive, or off-brand.'
    ),
    'helpfulness_system': 'You are a strict evaluator of support response helpfulness.',
    'helpfulness': (
        'Evaluate whether the response is helpful for the user question.\n\n'
        'User question:\n{user_input}\n\n'
        'Reference answer:\n{ideal}\n\n'
        'Actual response:\n{actual}\n\n'
        'Return exactly two lines in this format:\n'
        'LABEL: <Good|Acceptable|Poor>\n'
        'REASONING: <one sentence>\n\n'
        'GOOD: Gives the right next step or resolution with the key details.\n'
        'ACCEPTABLE: Directionally right but incomplete or vague.\n'
        'POOR: Misleading, unhelpful, or misses the main user need.'
    ),
}

experiment_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts=JUDGE_PROMPTS,
    limit_n=LIMIT_N_CASES,
    arize_client=arize_client,
    dataset_id=DATASET_ID,
)
arize_client = experiment_bundle['client']
DATASET_ID = experiment_bundle['dataset_id']
tasks = experiment_bundle['tasks']

print('Running routed copilot experiment with evaluators...')
run_v2_routed = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix='v2-routed-copilot',
    task=tasks['task_v2_routed'],
    evaluators=experiment_bundle['build_evaluators']('v2_routed'),
    concurrency=1,
)

print(f"Scored routed experiment ID: {run_v2_routed['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_routed['results_df'])}")


Running routed copilot experiment with evaluators...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/11/26 04:54 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/40 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.
Scored routed experiment ID: RXhwZXJpbWVudDo3NjEyOTpYcjJP
Rows evaluated: 10
